<!-- track-identity-card -->
# Portability check on the recorded sessions

| | |
|---|---|
| Pipeline step | `11_user_portability.ipynb` |
| Manuscript section | 4.5 |
| Copied from | `notebooks/NB14_user_pipeline_v1.ipynb` |
| Source sha256 | `9f75982a89681585c11db4f679098d57` |

**Reads**

- `data/raw/user_data/standardized/*monza*_lap*.parquet` (from 10)
- `data/features/monza_corners_v3.parquet` (from 02)
- `data/features/driver_corner_matrix_monza.parquet` (from 03)

**Writes**

- `data/features/driver_corner_matrix_monza_efe.parquet`
- `data/features/efe_corner_details_monza.parquet`
- `data/fingerprints/fingerprint_monza_efe.parquet`

Runs the segmentation and the 19 metrics of the main pipeline over the recorded sessions, producing the single-record matrix that 08 reads for the portability check of Section 4.5. Requires 02, 03 and 10 to have run.

> Copied from the working notebook named above. Two changes were made to it: this identity card and the bootstrap cell that follows it were added, and the hard-coded data paths were replaced with the root that the bootstrap cell resolves. The analysis code is unchanged.


In [ ]:
# track-config-bootstrap
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c)); break
else:
    raise RuntimeError("track/config.py not found. Run from inside the repository, or set TRACK_ROOT.")
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


# NB14 — User Data Pipeline: Efe Monza Fingerprint + Cluster Positioning

**Amac:** Efe'nin Monza turlarini ACGym pipeline'indan gecir, 19-metrik fingerprint hesapla, T1 kume uzayina yerlestir.

**Inputlar:**
- `monza_corners_v3.parquet` — 37 viraj tanimi
- `driver_corner_matrix_monza.parquet` — T1 popülasyon (18 suruku)
- `fingerprint_cross_track.parquet` + `cluster_profiles.json`
- 11 Efe Monza _lap parquet dosyasi

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import savgol_filter
import json
import warnings
warnings.filterwarnings('ignore')

PROJECT      = TRACK_ROOT
FEATURES_DIR = PROJECT / "data" / "features"
FP_DIR       = PROJECT / "data" / "fingerprints"
FIG_DIR      = PROJECT / "results" / "figures"
STD_DIR      = PROJECT / "data" / "raw" / "user_data" / "standardized"

# Segmentasyon parametreleri (NB08 v4 ile ayni)
SEARCH_BACK_M      = 300
SEARCH_FWD_M       = 200
TRAIL_BRAKE_ZONE   = 0.40
TRAIL_MIN_PRESSURE = 0.03
MIN_APEX_SPEED     = 20.0   # km/h
MIN_ENTRY_SPEED    = 50.0   # km/h
MC_RADIUS_M        = 15
EXIT_FALLBACK_M    = 50

# 19-metrik boyut haritasi (NB09 v3 ile ayni)
DIMENSIONS = {
    'B1_Hiz': {
        'label': 'Hiz Yonetimi',
        'metrics': ['mean_apex_speed', 'speed_loss_eff', 'mean_mc_speed_ratio',
                     'mean_mc_lateral', 'mean_cex_accel_rate']
    },
    'B2_Frenleme': {
        'label': 'Frenleme Stili',
        'metrics': ['mean_brake_pressure', 'trail_braking_ratio', 'mean_trail_pressure',
                     'mean_slb_dist', 'mean_slb_decel', 'mean_ce_brake_turnin']
    },
    'B3_Strateji': {
        'label': 'Surus Stratejisi',
        'metrics': ['mean_coasting_dist', 'mean_cex_throttle_lag',
                     'pct_lift_coast', 'pct_flat_out']
    },
    'B4_Tutarlilik': {
        'label': 'Tutarlilik',
        'metrics': ['apex_speed_std', 'exit_speed_std',
                     'speed_loss_eff_std', 'braking_dist_std']
    },
}
DIM_NAMES = list(DIMENSIONS.keys())

ALL_METRICS = []
for cfg in DIMENSIONS.values():
    ALL_METRICS.extend(cfg['metrics'])

print(f"Config OK — {len(ALL_METRICS)} metrik, {len(DIM_NAMES)} boyut")
print(f"Efe Monza laps: {len(list(STD_DIR.glob('*monza*_lap*.parquet')))} dosya")

## 1. Kolon Adaptoru
Efe formatini ACGym formatina donusturur.

In [ ]:
def adapt_efe_to_acgym(df_efe):
    """
    Efe _lap parquet -> ACGym uyumlu DataFrame.
    Kolon isimleri + birim donusumleri.
    """
    df = df_efe.copy()
    
    # Birim donusumleri
    df['brakeStatus'] = df['brake'] / 100.0       # 0-100 -> 0-1
    df['accStatus']   = df['throttle'] / 100.0     # 0-100 -> 0-1
    df['speed_kmh']   = df['speed'] * 3.6          # m/s -> km/h
    
    # Kolon yeniden adlandirma
    rename_map = {
        'lap_distance': 'LapDist',
        'steer':        'steerAngle',
        'rpm':          'RPM',
        'gear':         'actualGear',
        'g_lat':        'accelX',
        'g_lon':        'accelY',
    }
    df = df.rename(columns=rename_map)
    
    return df

# Test
sample = pd.read_parquet(sorted(STD_DIR.glob("*monza*_lap0.parquet"))[0])
adapted = adapt_efe_to_acgym(sample)
print(f"Adapted kolonlar ({len(adapted.columns)}):")
print(f"  brakeStatus: {adapted['brakeStatus'].min():.2f} - {adapted['brakeStatus'].max():.2f}")
print(f"  accStatus:   {adapted['accStatus'].min():.2f} - {adapted['accStatus'].max():.2f}")
print(f"  speed_kmh:   {adapted['speed_kmh'].mean():.1f} km/h")
print(f"  LapDist:     {adapted['LapDist'].min():.0f} - {adapted['LapDist'].max():.0f} m")
print(f"  steerAngle:  {adapted['steerAngle'].min():.0f} - {adapted['steerAngle'].max():.0f} deg")

## 2. Veri Yukleme
Monza viraj tanimlari + tum Efe Monza turlari.

In [ ]:
# --- Monza corners_v3 ---
corners_v3 = pd.read_parquet(FEATURES_DIR / "monza_corners_v3.parquet")
print(f"Monza virajlari: {len(corners_v3)} viraj")
print(f"Kolonlar: {list(corners_v3.columns)}")
print(f"\nViraj apex mesafeleri (m):")
for _, row in corners_v3.iterrows():
    cid = row.get('corner_id', '?')
    apex = row.get('apex_dist', row.get('apex_lapdist', '?'))
    diff = row.get('difficulty_score', '?')
    print(f"  Corner {cid}: apex={apex:.0f}m, difficulty={diff}")

# --- Efe Monza laps ---
monza_laps = sorted(STD_DIR.glob("*monza*_lap*.parquet"))
print(f"\nEfe Monza turlari: {len(monza_laps)} dosya")

efe_laps = []
for f in monza_laps:
    df = adapt_efe_to_acgym(pd.read_parquet(f))
    car = f.stem.split("_monza_")[0].replace("user_monza_", "")
    lap_num = f.stem.split("_lap")[-1]
    df['source_file'] = f.name
    df['car'] = car
    df['lap_num'] = int(lap_num)
    efe_laps.append(df)
    print(f"  {f.name}: {len(df)} satir, speed={df['speed_kmh'].mean():.1f} km/h, car={car}")

print(f"\nToplam: {len(efe_laps)} tur yuklendi")

## 3. Core Pipeline Fonksiyonlari
NB08 v4 + NB09 v3 fonksiyonlari (degisiklik yok).

In [ ]:
def _empty_seg(apex_lapdist, apex_speed=np.nan):
    keys = [
        'seg_entry_dist', 'seg_apex_dist', 'seg_exit_dist',
        'seg_entry_speed', 'seg_apex_speed', 'seg_exit_speed',
        'seg_exit_method',
        'seg_braking_dist', 'seg_trail_braking', 'seg_trail_pressure',
        'seg_avg_brake_pressure', 'seg_coasting_dist',
        'seg_speed_loss_eff', 'seg_corner_width',
        'phase_slb_dist', 'phase_slb_decel_rate',
        'phase_ce_dist', 'phase_ce_brake_at_turnin',
        'phase_mc_speed_ratio', 'phase_mc_lateral_signal',
        'phase_cex_throttle_lag', 'phase_cex_accel_rate',
        'phase_turnin_method',
    ]
    result = {k: np.nan for k in keys}
    result['seg_apex_dist'] = float(apex_lapdist)
    result['seg_apex_speed'] = float(apex_speed) if not np.isnan(apex_speed) else np.nan
    result['seg_exit_method'] = 'skipped'
    result['seg_entry_valid'] = False
    result['seg_trail_braking'] = False
    return result


def votes_to_confidence(n):
    return {0: 0.0, 1: 0.3, 2: 0.6, 3: 0.85, 4: 1.0}.get(int(n), 0.5)


def segment_corner(df_lap, apex_lapdist, difficulty):
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values   if 'accStatus'   in df_lap.columns else np.zeros(len(df_lap))
    dist  = df_lap['LapDist'].values

    apex_idx = int(np.argmin(np.abs(dist - apex_lapdist)))
    apex_speed = float(speed[apex_idx])

    back_limit = max(0, apex_idx - int(SEARCH_BACK_M / 2))
    entry_idx  = apex_idx
    for j in range(apex_idx - 1, back_limit, -1):
        if brake[j] < 0.05 and speed[j] > speed[apex_idx]:
            entry_idx = j
            break

    entry_speed = float(speed[entry_idx])
    if entry_speed < MIN_ENTRY_SPEED or apex_speed < MIN_APEX_SPEED:
        return _empty_seg(apex_lapdist, apex_speed)

    # EXIT: 3-katmanli tespit (v4)
    fwd_limit = min(len(speed) - 1, apex_idx + int(SEARCH_FWD_M / 2))
    exit_idx    = apex_idx
    exit_method = 'none'

    for j in range(apex_idx + 1, fwd_limit):
        if acc[j] > 0.3 and speed[j] > apex_speed:
            exit_idx = j
            exit_method = 'throttle_full'
            break

    if exit_idx == apex_idx:
        for j in range(apex_idx + 1, fwd_limit):
            if acc[j] > 0.3:
                exit_idx = j
                exit_method = 'throttle_only'
                break

    if exit_idx == apex_idx:
        fb_target = dist[apex_idx] + EXIT_FALLBACK_M
        fb_idx = int(np.argmin(np.abs(dist - fb_target)))
        exit_idx = min(fb_idx, fwd_limit)
        exit_method = 'distance_fallback'

    exit_speed = float(speed[exit_idx])

    entry_dist_v = float(dist[entry_idx])
    exit_dist_v  = float(dist[exit_idx])
    braking_dist = float(dist[apex_idx] - dist[entry_idx]) if entry_idx < apex_idx else 0.0
    speed_loss_eff = (apex_speed / entry_speed) if entry_speed > 0 else 0.0

    coast_start = apex_idx
    for j in range(apex_idx, fwd_limit):
        if brake[j] < 0.05:
            coast_start = j
            break
    coast_end = coast_start
    for j in range(coast_start, fwd_limit):
        if acc[j] > 0.1:
            coast_end = j
            break
    coasting_dist = float(dist[coast_end] - dist[coast_start]) if coast_end > coast_start else 0.0

    if braking_dist > 10:
        trail_zone_start = entry_idx + int((apex_idx - entry_idx) * (1 - TRAIL_BRAKE_ZONE))
        trail_pressures  = brake[trail_zone_start:apex_idx]
        trail_active     = trail_pressures.mean() > TRAIL_MIN_PRESSURE if len(trail_pressures) > 0 else False
        trail_pressure   = float(trail_pressures.mean()) if len(trail_pressures) > 0 else 0.0
    else:
        trail_active   = False
        trail_pressure = 0.0

    avg_brake = float(brake[entry_idx:apex_idx].mean()) if apex_idx > entry_idx else 0.0

    return {
        'seg_entry_dist': entry_dist_v, 'seg_apex_dist': float(dist[apex_idx]),
        'seg_exit_dist': exit_dist_v,
        'seg_entry_speed': entry_speed, 'seg_apex_speed': apex_speed,
        'seg_exit_speed': exit_speed,
        'seg_exit_method': exit_method,
        'seg_braking_dist': braking_dist, 'seg_trail_braking': trail_active,
        'seg_trail_pressure': trail_pressure, 'seg_avg_brake_pressure': avg_brake,
        'seg_coasting_dist': coasting_dist, 'seg_speed_loss_eff': speed_loss_eff,
        'seg_corner_width': exit_dist_v - entry_dist_v,
        'seg_entry_valid': True,
    }


def segment_corner_phases(df_lap, seg, steer_info):
    mode, steer_col = steer_info

    if not seg.get('seg_entry_valid', False) or pd.isna(seg.get('seg_entry_dist')):
        return {
            'phase_slb_dist': np.nan, 'phase_slb_decel_rate': np.nan,
            'phase_ce_dist': np.nan, 'phase_ce_brake_at_turnin': np.nan,
            'phase_mc_speed_ratio': np.nan, 'phase_mc_lateral_signal': np.nan,
            'phase_cex_throttle_lag': np.nan, 'phase_cex_accel_rate': np.nan,
            'phase_turnin_method': 'skipped',
        }

    dist  = df_lap['LapDist'].values
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values if 'accStatus' in df_lap.columns else np.zeros(len(df_lap))

    entry_dist = seg['seg_entry_dist']
    apex_dist  = seg['seg_apex_dist']
    exit_dist  = seg['seg_exit_dist']

    entry_idx = int(np.argmin(np.abs(dist - entry_dist)))
    apex_idx  = int(np.argmin(np.abs(dist - apex_dist)))
    exit_idx  = int(np.argmin(np.abs(dist - exit_dist)))

    # Turn-in noktasi tespiti
    turnin_idx = entry_idx + int((apex_idx - entry_idx) * 0.4)
    turnin_method = 'speed_proxy_40pct'

    if mode == 'steer' and steer_col in df_lap.columns:
        steer = np.abs(df_lap[steer_col].values)
        region = steer[entry_idx:apex_idx]
        if len(region) > 5:
            wlen = min(11, len(region))
            if wlen % 2 == 0:
                wlen -= 1
            if wlen >= 3:
                smoothed = savgol_filter(region, wlen, min(2, wlen - 1))
            else:
                smoothed = region
            threshold = smoothed.max() * 0.15
            for k, val in enumerate(smoothed):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'steer_threshold'
                    break

    elif mode == 'g_lat' and steer_col in df_lap.columns:
        glat = np.abs(df_lap[steer_col].values)
        region = glat[entry_idx:apex_idx]
        if len(region) > 5:
            threshold = region.max() * 0.20
            for k, val in enumerate(region):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'g_lat_threshold'
                    break

    turnin_idx = max(entry_idx + 1, min(turnin_idx, apex_idx - 1))

    mc_start = int(np.argmin(np.abs(dist - (apex_dist - MC_RADIUS_M))))
    mc_end   = int(np.argmin(np.abs(dist - (apex_dist + MC_RADIUS_M))))
    mc_start = max(mc_start, turnin_idx)
    mc_end   = min(mc_end, exit_idx)

    slb_dist = float(dist[turnin_idx] - dist[entry_idx]) if turnin_idx > entry_idx else 0.0
    slb_speed_drop = float(speed[entry_idx] - speed[turnin_idx])
    slb_decel = slb_speed_drop / max(slb_dist, 1.0) * (1000.0 / 3600.0)

    ce_dist = float(dist[mc_start] - dist[turnin_idx]) if mc_start > turnin_idx else 0.0
    ce_brake_at_turnin = float(brake[turnin_idx]) if turnin_idx < len(brake) else 0.0

    mc_speeds = speed[mc_start:mc_end+1]
    mc_speed_ratio = float(mc_speeds.min() / mc_speeds.max()) if len(mc_speeds) > 1 and mc_speeds.max() > 0 else 1.0

    mc_lateral = 0.0
    if (mode == 'g_lat' or mode == 'steer') and steer_col in df_lap.columns:
        mc_lateral = float(np.abs(df_lap[steer_col].values[mc_start:mc_end+1]).mean())

    throttle_lag = 0.0
    for j in range(apex_idx, exit_idx):
        if acc[j] > 0.3:
            throttle_lag = float(dist[j] - dist[apex_idx])
            break

    cex_region = speed[apex_idx:exit_idx+1]
    if len(cex_region) > 1:
        cex_accel = float(cex_region[-1] - cex_region[0]) / max(float(dist[exit_idx] - dist[apex_idx]), 1.0)
    else:
        cex_accel = 0.0

    return {
        'phase_slb_dist': slb_dist,
        'phase_slb_decel_rate': slb_decel,
        'phase_ce_dist': ce_dist,
        'phase_ce_brake_at_turnin': ce_brake_at_turnin,
        'phase_mc_speed_ratio': mc_speed_ratio,
        'phase_mc_lateral_signal': mc_lateral,
        'phase_cex_throttle_lag': throttle_lag,
        'phase_cex_accel_rate': cex_accel,
        'phase_turnin_method': turnin_method,
    }


def detect_steer_column(df):
    cols = {c: c.lower() for c in df.columns}
    for c, cl in cols.items():
        if any(k in cl for k in ['steerangle', 'steering_angle', 'steer_angle', 'wheel_angle']):
            return ('steer', c)
    for c, cl in cols.items():
        if 'steer' in cl and 'error' not in cl:
            return ('steer', c)
    for c, cl in cols.items():
        if any(k in cl for k in ['g_lat', 'glat', 'lateral_g', 'accg_y', 'accel_lat']):
            return ('g_lat', c)
    return ('speed_proxy', None)


def wavg(values, weights):
    v    = pd.to_numeric(pd.Series(values), errors='coerce').values
    w    = np.array(weights, dtype=float)
    mask = ~np.isnan(v) & (w > 0)
    return float(np.average(v[mask], weights=w[mask])) if mask.sum() > 0 else np.nan


print("Core fonksiyonlar yuklendi")
# Steer tespiti (Efe verisi icin)
steer_info = detect_steer_column(efe_laps[0])
print(f"Steer modu: {steer_info}")

## 4. Tur-Bazli Viraj Segmentasyonu
Her Efe turu icin 37 viraji ayri ayri isle, sonra birlestir.

In [ ]:
all_corner_results = []

for lap_idx, df_lap in enumerate(efe_laps):
    car  = df_lap['car'].iloc[0]
    lnum = df_lap['lap_num'].iloc[0]
    
    lap_corners = []
    for _, crow in corners_v3.iterrows():
        apex_d = float(crow.get('apex_dist', crow.get('apex_lapdist', 0)))
        diff   = float(crow.get('difficulty_score', 0.5))
        
        seg = segment_corner(df_lap, apex_d, diff)
        phase = segment_corner_phases(df_lap, seg, steer_info)
        seg.update(phase)
        
        # Corner meta ekle
        seg['corner_id'] = crow.get('corner_id', lap_idx)
        seg['n_votes']   = crow.get('n_votes', 3)
        seg['confidence'] = votes_to_confidence(seg.get('n_votes', 3))
        
        # character_class (varsa)
        if 'character_class' in crow.index:
            seg['character_class'] = crow['character_class']
        
        # Lap meta
        seg['lap_idx'] = lap_idx
        seg['car'] = car
        seg['lap_num'] = lnum
        
        # Confidence guncellemeleri
        if not seg.get('seg_entry_valid', False):
            seg['confidence'] = 0.0
        if seg.get('seg_apex_speed', 0) < MIN_APEX_SPEED:
            seg['confidence'] = 0.0
        if seg.get('seg_entry_speed', 0) < seg.get('seg_apex_speed', 0):
            seg['confidence'] = 0.0
        
        lap_corners.append(seg)
    
    valid = sum(1 for c in lap_corners if c.get('confidence', 0) > 0)
    print(f"  Tur {lap_idx:2d} ({car}, lap{lnum}): {valid}/{len(lap_corners)} viraj gecerli")
    all_corner_results.extend(lap_corners)

corners_df = pd.DataFrame(all_corner_results)
valid_total = (corners_df['confidence'] > 0).sum()
print(f"\nToplam: {len(corners_df)} viraj-tur kombinasyonu, {valid_total} gecerli")
print(f"Tur basina ortalama gecerli viraj: {valid_total / len(efe_laps):.1f}")

## 5. Efe Driver Matrix
Tum turlardan viraj metriklerini agrega et — coklu tur avantaji (daha iyi B4 std).

In [ ]:
w = corners_df['confidence'].fillna(0).values
valid = corners_df[corners_df['confidence'] > 0]

if len(valid) == 0:
    raise ValueError("Hicbir gecerli viraj bulunamadi!")

efe_matrix = {
    'driver_id': 'EFE_USER', 'track': 'monza',
    'n_corners_total': len(corners_df),
    'n_corners_valid': len(valid),
    'n_laps': len(efe_laps),
    # Mevcut metrikler
    'mean_apex_speed':     wavg(corners_df['seg_apex_speed'], w),
    'mean_entry_speed':    wavg(corners_df['seg_entry_speed'], w),
    'mean_exit_speed':     wavg(corners_df['seg_exit_speed'], w),
    'speed_loss_eff':      wavg(corners_df['seg_speed_loss_eff'], w),
    'mean_braking_dist':   wavg(corners_df['seg_braking_dist'], w),
    'mean_brake_pressure': wavg(corners_df['seg_avg_brake_pressure'], w),
    'mean_coasting_dist':  wavg(corners_df['seg_coasting_dist'], w),
    'trail_braking_ratio': float(valid['seg_trail_braking'].mean()),
    'mean_trail_pressure': wavg(corners_df['seg_trail_pressure'], w),
    'apex_speed_std':      float(valid['seg_apex_speed'].std()),
    'braking_dist_std':    float(valid['seg_braking_dist'].std()),
    'exit_speed_std':      float(valid['seg_exit_speed'].std()),
    'speed_loss_eff_std':  float(valid['seg_speed_loss_eff'].std()),
    # character_class metrikleri
    'pct_heavy_braking':   float((valid['character_class'] == 'heavy_braking').mean()) if 'character_class' in valid.columns else np.nan,
    'pct_trail_braking':   float((valid['character_class'] == 'trail_braking').mean()) if 'character_class' in valid.columns else np.nan,
    'pct_lift_coast':      float((valid['character_class'] == 'lift_coast').mean()) if 'character_class' in valid.columns else np.nan,
    'pct_flat_out':        float((valid['character_class'] == 'flat_out').mean()) if 'character_class' in valid.columns else np.nan,
    # Faz metrikleri (v3)
    'mean_slb_dist':         wavg(corners_df['phase_slb_dist'], w),
    'mean_slb_decel':        wavg(corners_df['phase_slb_decel_rate'], w),
    'mean_ce_dist':          wavg(corners_df['phase_ce_dist'], w),
    'mean_ce_brake_turnin':  wavg(corners_df['phase_ce_brake_at_turnin'], w),
    'mean_mc_speed_ratio':   wavg(corners_df['phase_mc_speed_ratio'], w),
    'mean_mc_lateral':       wavg(corners_df['phase_mc_lateral_signal'], w),
    'mean_cex_throttle_lag': wavg(corners_df['phase_cex_throttle_lag'], w),
    'mean_cex_accel_rate':   wavg(corners_df['phase_cex_accel_rate'], w),
}

print("=== EFE DRIVER MATRIX ===")
print(f"Gecerli virajlar: {efe_matrix['n_corners_valid']}/{efe_matrix['n_corners_total']}")
print(f"Turlar: {efe_matrix['n_laps']}")
print(f"\n--- 19 Metrik ---")
for m in ALL_METRICS:
    val = efe_matrix.get(m, np.nan)
    print(f"  {m:30s}: {val:.4f}" if not np.isnan(val) else f"  {m:30s}: NaN")

## 6. T1 Populasyonla Karsilastirma
Efe fingerprint + T1 populasyon fingerprint yan yana.

In [ ]:
# --- T1 Monza matrix ---
t1_monza = pd.read_parquet(FEATURES_DIR / "driver_corner_matrix_monza.parquet")
print(f"T1 Monza: {len(t1_monza)} suruku")

# Efe'yi T1 matrix'e ekle
efe_row = pd.DataFrame([efe_matrix])
combined = pd.concat([t1_monza, efe_row], ignore_index=True)

# --- Per-track fingerprint (Min-Max normalization) ---
def compute_fingerprint_single_track(df_track):
    rows = []
    for _, drv in df_track.iterrows():
        row = {'driver_id': drv['driver_id']}
        for dim, cfg in DIMENSIONS.items():
            vals = []
            for m in cfg['metrics']:
                if m in drv.index and pd.notna(drv[m]):
                    vals.append(float(drv[m]))
            row[dim] = np.mean(vals) if vals else np.nan
        rows.append(row)
    
    fp = pd.DataFrame(rows)
    
    for dim in DIM_NAMES:
        col = fp[dim].copy()
        if dim == 'B4_Tutarlilik':
            col = 1.0 - col  # Ters: yuksek std = dusuk tutarlilik
        mn, mx = col.min(), col.max()
        if mx > mn:
            fp[dim] = (col - mn) / (mx - mn)
        else:
            fp[dim] = 0.5
    
    return fp

fp_combined = compute_fingerprint_single_track(combined)
fp_combined['driver_id'] = combined['driver_id'].values

# Efe'nin fingerprint'ini ayir
efe_fp = fp_combined[fp_combined['driver_id'] == 'EFE_USER'].iloc[0]
t1_fps = fp_combined[fp_combined['driver_id'] != 'EFE_USER']

print(f"\n=== EFE FINGERPRINT (Monza, T1 populasyonla normalize) ===")
for dim in DIM_NAMES:
    val = efe_fp[dim]
    bar = '#' * int(val * 30)
    print(f"  {dim:20s}: {val:.3f}  |{bar}")

print(f"\n--- T1 Populasyon Ortalamalari ---")
for dim in DIM_NAMES:
    mean = t1_fps[dim].mean()
    std  = t1_fps[dim].std()
    efe_val = efe_fp[dim]
    delta = efe_val - mean
    arrow = "^" if delta > 0 else "v"
    print(f"  {dim:20s}: T1={mean:.3f}+/-{std:.3f}  Efe={efe_val:.3f}  ({arrow}{abs(delta):.3f})")

## 7. Kume Konumlama
Efe'yi T1 k=3 kume uzayina yerlestir.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# T1 populasyonu kumele (NB10A ile ayni)
X_t1 = t1_fps[DIM_NAMES].fillna(0.5).values
km = KMeans(n_clusters=3, random_state=42, n_init=20)
t1_labels = km.fit_predict(X_t1)
t1_fps = t1_fps.copy()
t1_fps['cluster'] = t1_labels

# Kume profilleri
print("=== T1 KUME PROFILLERI ===")
for ci in range(3):
    members = t1_fps[t1_fps['cluster'] == ci]
    center = {d: members[d].mean() for d in DIM_NAMES}
    dominant = max(DIM_NAMES, key=lambda d: center[d])
    n = len(members)
    print(f"  Cluster {ci}: {n} suruku, baskin={dominant} ({center[dominant]:.3f})")
    for d in DIM_NAMES:
        print(f"    {d}: {center[d]:.3f}")

# Efe'yi en yakin kumeye ata
efe_vec = np.array([efe_fp[d] for d in DIM_NAMES]).reshape(1, -1)
efe_cluster = km.predict(efe_vec)[0]
distances = km.transform(efe_vec)[0]

print(f"\n=== EFE KUME ATAMASI ===")
print(f"En yakin kume: Cluster {efe_cluster}")
print(f"Mesafeler: {', '.join(f'C{i}={d:.3f}' for i, d in enumerate(distances))}")

# Gap analizi: Efe vs kume merkezi
cluster_center = km.cluster_centers_[efe_cluster]
print(f"\n=== GAP ANALIZI (Efe vs Cluster {efe_cluster} merkezi) ===")
gaps = {}
for i, dim in enumerate(DIM_NAMES):
    gap = efe_fp[dim] - cluster_center[i]
    gaps[dim] = gap
    direction = "GUCLU" if gap > 0.05 else "ZAYIF" if gap < -0.05 else "ORTALAMA"
    print(f"  {dim:20s}: gap={gap:+.3f}  [{direction}]")

## 8. Radar Chart: Efe vs T1 Kumeleri

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import matplotlib
matplotlib.rcParams['figure.facecolor'] = 'white'

# Radar chart
angles = np.linspace(0, 2 * np.pi, len(DIM_NAMES), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# T1 kume merkezleri
colors = ['#2196F3', '#FF9800', '#4CAF50']
for ci in range(3):
    center = km.cluster_centers_[ci].tolist() + [km.cluster_centers_[ci][0]]
    members = t1_fps[t1_fps['cluster'] == ci]
    n = len(members)
    ax.plot(angles, center, 'o-', color=colors[ci], linewidth=1.5, alpha=0.6,
            label=f'C{ci} ({n} suruku)', markersize=4)
    ax.fill(angles, center, alpha=0.08, color=colors[ci])

# Efe
efe_vals = [efe_fp[d] for d in DIM_NAMES] + [efe_fp[DIM_NAMES[0]]]
ax.plot(angles, efe_vals, 'D-', color='#E91E63', linewidth=2.5,
        label='EFE', markersize=7, zorder=10)
ax.fill(angles, efe_vals, alpha=0.15, color='#E91E63')

# Eksen ayarlari
dim_labels = [DIMENSIONS[d]['label'] for d in DIM_NAMES]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8, alpha=0.5)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.set_title('Efe vs T1 Kume Merkezleri (Monza)', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(FIG_DIR / "efe_vs_t1_radar.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nRadar chart kaydedildi: results/figures/efe_vs_t1_radar.png")

## 9. Sonuc Kaydi

In [ ]:
# Efe matrix kaydet
efe_df = pd.DataFrame([efe_matrix])
out_path = FEATURES_DIR / "driver_corner_matrix_monza_efe.parquet"
efe_df.to_parquet(out_path, index=False)
print(f"Efe matrix kaydedildi: {out_path.name}")

# Efe fingerprint kaydet
efe_fp_df = fp_combined[fp_combined['driver_id'] == 'EFE_USER'].copy()
efe_fp_df['cluster'] = int(efe_cluster)
efe_fp_df['cluster_distance'] = float(distances[efe_cluster])
fp_out = FP_DIR / "fingerprint_monza_efe.parquet"
efe_fp_df.to_parquet(fp_out, index=False)
print(f"Efe fingerprint kaydedildi: {fp_out.name}")

# Per-corner-per-lap detay kaydet
corners_out = FEATURES_DIR / "efe_corner_details_monza.parquet"
corners_df.to_parquet(corners_out, index=False)
print(f"Efe viraj detaylari kaydedildi: {corners_out.name}")

print(f"\n{'='*50}")
print(f"  OZET")
print(f"{'='*50}")
print(f"  Turlar: {len(efe_laps)} (Monza, {corners_df['car'].nunique()} farkli arac)")
print(f"  Gecerli virajlar: {efe_matrix['n_corners_valid']}/{efe_matrix['n_corners_total']}")
print(f"  Kume atamasi: Cluster {efe_cluster}")
for dim in DIM_NAMES:
    print(f"  {dim}: {efe_fp[dim]:.3f}")

In [ ]:
# ============================================================
# ADIM A: Car extraction fix + Binary brake diagnostik
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np

STD_DIR = TRACK_ROOT / "data" / "raw" / "user_data" / "standardized"
T1_DIR  = TRACK_ROOT / "data" / "processed" / "tier1_small_3t1c"

# --- 1. Car extraction fix ---
print("=== CAR EXTRACTION ===")
monza_laps = sorted(STD_DIR.glob("*monza*_lap*.parquet"))

def extract_car_from_filename(fname):
    """user_monza_{car_short}_monza-{car_full}-{session}-{ts}_lap{N}"""
    stem = fname.stem  # dosya adi (.parquet olmadan)
    # "monza-" den sonraki kisim: car_full-session-timestamp
    if "monza-" in stem:
        after_track = stem.split("monza-")[1]  # "nissan_gt_r_gt3_2018-6-2026..."
        # _lap ile kes
        before_lap = after_track.rsplit("_lap", 1)[0]  # "nissan_gt_r_gt3_2018-6-2026.02.15-23.18.56"
        # Ilk tire grubuna kadar = car
        parts = before_lap.split("-")
        # Car name: tire ile ayrilmis, rakamla baslayan kisma kadar
        car_parts = []
        for p in parts:
            if p and p[0].isdigit() and len(p) <= 2:
                break  # session number'a ulastik
            car_parts.append(p)
        return "_".join(car_parts)
    return "unknown"

car_counts = {}
for f in monza_laps:
    car = extract_car_from_filename(f)
    car_counts[car] = car_counts.get(car, 0) + 1
    
print("Monza araclari:")
for car, count in sorted(car_counts.items()):
    print(f"  {car}: {count} tur")

# --- 2. Binary brake analizi ---
print("\n=== BINARY BRAKE ANALIZI ===")

# Efe brake dagilimi
efe_all_brake = []
for f in monza_laps:
    df = pd.read_parquet(f)
    efe_all_brake.extend(df["brake"].values.tolist())

efe_brake = np.array(efe_all_brake)
unique_vals = np.unique(efe_brake)
print(f"Efe brake unique degerler: {len(unique_vals)}")
if len(unique_vals) <= 20:
    print(f"  Degerler: {unique_vals}")
else:
    print(f"  Min={efe_brake.min():.1f}, Max={efe_brake.max():.1f}")
    print(f"  Ilk 20: {unique_vals[:20]}")

# 0 ve 100 yuzdeleri
pct_zero = (efe_brake == 0).mean() * 100
pct_full = (efe_brake == 100).mean() * 100
pct_partial = ((efe_brake > 0) & (efe_brake < 100)).mean() * 100
print(f"\n  %0 (fren yok):    {pct_zero:.1f}%")
print(f"  %100 (tam fren):  {pct_full:.1f}%")
print(f"  Ara deger:        {pct_partial:.1f}%")

# ACGym brake dagilimi
print("\n--- ACGym karsilastirma ---")
t1_sample = pd.read_parquet(sorted(T1_DIR.glob("*monza*bmw*.parquet"))[0])
acgym_brake = t1_sample["brakeStatus"].values
acgym_zero = (acgym_brake < 0.01).mean() * 100
acgym_full = (acgym_brake > 0.99).mean() * 100
acgym_partial = ((acgym_brake >= 0.01) & (acgym_brake <= 0.99)).mean() * 100
print(f"  %0 (fren yok):    {acgym_zero:.1f}%")
print(f"  %100 (tam fren):  {acgym_full:.1f}%")
print(f"  Ara deger:        {acgym_partial:.1f}%")

# --- 3. Trail braking etkisi ---
print("\n=== TRAIL BRAKING ETKISi ===")
print("Trail braking tespiti: fren bolgesinin son %40'inda ortalama basinc > 0.03")
print(f"Efe trail_braking_ratio:    {0.0667:.3f} (cok dusuk)")
print(f"T1 populasyon ortalamasi:   ~0.35-0.45 (beklenen)")
print("\nNeden dusuk? Binary fren sinyali sebebiyle:")
print("  - ACGym: fren yavas yavas azalir (0.8 -> 0.5 -> 0.2 -> 0)")
print("  - Efe:   fren ani kesilir (100 -> 0, ara deger YOK)")
print("  - Trail zone'da basinc = 0 cikiyor cunku fren zaten birakildiktan sonra")

In [ ]:
# ============================================================
# ADIM B: Cross-car fingerprint analizi (Monza)
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

STD_DIR = TRACK_ROOT / "data" / "raw" / "user_data" / "standardized"
FEATURES_DIR = TRACK_ROOT / "data" / "features"
FIG_DIR = TRACK_ROOT / "results" / "figures"

# NB14'ten corners_v3 ve fonksiyonlar zaten yukluyse devam et
# (ayni notebook icinde calistiriyorsan bu satirlari atla)

# --- 1. Per-car lap gruplama ---
def extract_car(fname):
    stem = fname.stem
    if "monza-" in stem:
        after = stem.split("monza-")[1]
        before_lap = after.rsplit("_lap", 1)[0]
        parts = before_lap.split("-")
        car_parts = []
        for p in parts:
            if p and p[0].isdigit() and len(p) <= 2:
                break
            car_parts.append(p)
        return "_".join(car_parts)
    return "unknown"

monza_files = sorted(STD_DIR.glob("*monza*_lap*.parquet"))
car_groups = {}
for f in monza_files:
    car = extract_car(f)
    if car not in car_groups:
        car_groups[car] = []
    car_groups[car].append(f)

print("Monza per-car gruplama:")
for car, files in sorted(car_groups.items()):
    print(f"  {car}: {len(files)} tur")

# --- 2. Per-car segmentasyon + matrix ---
car_matrices = {}
car_corner_details = {}

for car, files in sorted(car_groups.items()):
    laps = [adapt_efe_to_acgym(pd.read_parquet(f)) for f in files]
    
    all_segs = []
    for lap_df in laps:
        for _, crow in corners_v3.iterrows():
            apex_d = float(crow.get('apex_dist', 0))
            diff = float(crow.get('difficulty_score', 0.5))
            seg = segment_corner(lap_df, apex_d, diff)
            phase = segment_corner_phases(lap_df, seg, steer_info)
            seg.update(phase)
            seg['corner_id'] = crow.get('corner_id', 0)
            seg['n_votes'] = crow.get('n_votes', 3)
            seg['confidence'] = votes_to_confidence(seg.get('n_votes', 3))
            if 'character_class' in crow.index:
                seg['character_class'] = crow['character_class']
            if not seg.get('seg_entry_valid', False):
                seg['confidence'] = 0.0
            if seg.get('seg_apex_speed', 0) < MIN_APEX_SPEED:
                seg['confidence'] = 0.0
            all_segs.append(seg)
    
    cdf = pd.DataFrame(all_segs)
    w = cdf['confidence'].fillna(0).values
    valid = cdf[cdf['confidence'] > 0]
    
    if len(valid) == 0:
        print(f"  {car}: HICBIR GECERLI VIRAJ YOK — atlaniyor")
        continue
    
    mat = {
        'driver_id': f'EFE_{car}', 'track': 'monza', 'car': car,
        'n_laps': len(laps), 'n_corners_valid': len(valid),
        'mean_apex_speed': wavg(cdf['seg_apex_speed'], w),
        'speed_loss_eff': wavg(cdf['seg_speed_loss_eff'], w),
        'mean_mc_speed_ratio': wavg(cdf['phase_mc_speed_ratio'], w),
        'mean_mc_lateral': wavg(cdf['phase_mc_lateral_signal'], w),
        'mean_cex_accel_rate': wavg(cdf['phase_cex_accel_rate'], w),
        'mean_brake_pressure': wavg(cdf['seg_avg_brake_pressure'], w),
        'trail_braking_ratio': float(valid['seg_trail_braking'].mean()),
        'mean_trail_pressure': wavg(cdf['seg_trail_pressure'], w),
        'mean_slb_dist': wavg(cdf['phase_slb_dist'], w),
        'mean_slb_decel': wavg(cdf['phase_slb_decel_rate'], w),
        'mean_ce_brake_turnin': wavg(cdf['phase_ce_brake_at_turnin'], w),
        'mean_coasting_dist': wavg(cdf['seg_coasting_dist'], w),
        'mean_cex_throttle_lag': wavg(cdf['phase_cex_throttle_lag'], w),
        'pct_lift_coast': float((valid['character_class'] == 'lift_coast').mean()) if 'character_class' in valid.columns else np.nan,
        'pct_flat_out': float((valid['character_class'] == 'flat_out').mean()) if 'character_class' in valid.columns else np.nan,
        'apex_speed_std': float(valid['seg_apex_speed'].std()) if len(valid) > 1 else np.nan,
        'exit_speed_std': float(valid['seg_exit_speed'].std()) if len(valid) > 1 else np.nan,
        'speed_loss_eff_std': float(valid['seg_speed_loss_eff'].std()) if len(valid) > 1 else np.nan,
        'braking_dist_std': float(valid['seg_braking_dist'].std()) if len(valid) > 1 else np.nan,
    }
    car_matrices[car] = mat
    print(f"  {car}: {len(laps)} tur, {len(valid)} gecerli viraj, apex={mat['mean_apex_speed']:.1f} km/h")

# --- 3. Cross-car fingerprint ---
print("\n=== CROSS-CAR FINGERPRINT (ayni suruku, farkli arac) ===")
car_fps = {}
for car, mat in car_matrices.items():
    fp = {}
    for dim, cfg in DIMENSIONS.items():
        vals = [mat[m] for m in cfg['metrics'] if m in mat and not np.isnan(mat.get(m, np.nan))]
        fp[dim] = np.mean(vals) if vals else np.nan
    car_fps[car] = fp

# Raw (normalize edilmemis) degerler
print("\nRaw boyut degerleri (normalizasyon oncesi):")
print(f"{'Arac':<35s} {'B1_Hiz':>10s} {'B2_Fren':>10s} {'B3_Str':>10s} {'B4_Tut':>10s}")
print("-" * 75)
for car, fp in sorted(car_fps.items()):
    vals = [f"{fp[d]:.4f}" if not np.isnan(fp[d]) else "NaN" for d in DIM_NAMES]
    print(f"{car:<35s} {'  '.join(vals)}")

# Araclar arasi korelasyon
print("\n--- Boyut bazinda min/max/range ---")
for dim in DIM_NAMES:
    vals = [fp[dim] for fp in car_fps.values() if not np.isnan(fp[dim])]
    if vals:
        rng = max(vals) - min(vals)
        cv = np.std(vals) / np.mean(vals) * 100 if np.mean(vals) != 0 else 0
        print(f"  {dim:20s}: min={min(vals):.4f} max={max(vals):.4f} range={rng:.4f} CV={cv:.1f}%")

# --- 4. Radar chart: per-car ---
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(DIM_NAMES), endpoint=False).tolist()
angles += angles[:1]

car_colors = {
    'audi_r8_lms_evo': '#FF1744',
    'bmw_m6_gt3': '#2979FF',
    'lamborghini_huracan_gt3_evo': '#00E676',
    'nissan_gt_r_gt3_2018': '#FFD600',
}

# Normalize: tum araclari birlestirip min-max
all_raw = pd.DataFrame([car_fps[c] for c in car_fps])
for dim in DIM_NAMES:
    col = all_raw[dim].copy()
    if dim == 'B4_Tutarlilik':
        col = 1.0 - col
    mn, mx = col.min(), col.max()
    if mx > mn:
        all_raw[dim] = (col - mn) / (mx - mn)
    else:
        all_raw[dim] = 0.5

for i, (car, _) in enumerate(sorted(car_fps.items())):
    row = all_raw.iloc[i]
    vals = [row[d] for d in DIM_NAMES] + [row[DIM_NAMES[0]]]
    color = car_colors.get(car, f'C{i}')
    n_laps = car_matrices[car]['n_laps']
    short_name = car.replace("_gt3", "").replace("_evo", "").replace("_2018", "").replace("_lms", "")
    ax.plot(angles, vals, 'o-', color=color, linewidth=2, label=f'{short_name} ({n_laps}t)', markersize=6)
    ax.fill(angles, vals, alpha=0.08, color=color)

dim_labels = [DIMENSIONS[d]['label'] for d in DIM_NAMES]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
ax.set_title('Efe Cross-Car Fingerprint (Monza)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(FIG_DIR / "efe_cross_car_radar.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nKaydedildi: results/figures/efe_cross_car_radar.png")

In [ ]:
# ============================================================
# SAC REFERANS TESHIS - tek hucre
# Amac: SAC yapisini cikar; degenerate vs oran-uygun metrik adaylarini gor
# ============================================================
import pandas as pd, numpy as np
from pathlib import Path
from collections import Counter

PROJECT = TRACK_ROOT

# 1) SAC parquet dizinini otomatik bul
candidates = [
    PROJECT / "data" / "reference_sac",
    PROJECT / "data" / "processed" / "reference_sac",
    PROJECT / "data" / "acgym_sac",
    PROJECT / "data" / "raw" / "acgym_sac",
]
SAC_DIR = None
for c in candidates:
    n = len(list(c.rglob("*.parquet"))) if c.exists() else 0
    print(f"  aday: {c}  ->  {n} parquet")
    if n > 0 and SAC_DIR is None:
        SAC_DIR = c
if SAC_DIR is None:
    raise SystemExit("SAC parquet dizini bulunamadi - candidates listesini guncelle")
print(f"\nSAC_DIR = {SAC_DIR}")

sac_files = sorted(SAC_DIR.rglob("*.parquet"))
print(f"Toplam SAC parquet: {len(sac_files)}")

# 2) Pist envanteri (dosya adindan)
TRACK_TOKENS = ["monza","barcelona","red_bull","redbull","rbr","silverstone","indianapolis"]
track_counts = Counter()
for f in sac_files:
    name = f.name.lower()
    hit = next((t for t in TRACK_TOKENS if t in name), "OTHER")
    track_counts[hit] += 1
print("\nPist dagilimi (SAC):")
for t, n in track_counts.most_common():
    print(f"  {t:14s}: {n}")

# 3) Bir Monza SAC vs Monza T1 insan: davranis karsilastir
T1_DIR = PROJECT / "data" / "processed" / "tier1_small_3t1c"
def first_match(files, token):
    return next((f for f in files if token in f.name.lower()), None)

sac_monza = first_match(sac_files, "monza")
hum_monza = first_match(sorted(T1_DIR.glob("*.parquet")), "monza")
print(f"\nSAC ornegi  : {sac_monza.name if sac_monza else 'YOK'}")
print(f"Insan ornegi: {hum_monza.name if hum_monza else 'YOK'}")

def behav(df):
    cols = df.columns
    bcol = "brakeStatus" if "brakeStatus" in cols else ("brake" if "brake" in cols else None)
    tcol = next((c for c in ["accStatus","Gas","throttle"] if c in cols), None)
    scol = "speed_kmh" if "speed_kmh" in cols else ("speed" if "speed" in cols else None)
    r = {}
    if bcol is not None:
        b = df[bcol].astype(float); bmax = b.max()
        thr_zero = 0.01 * bmax if bmax > 0 else 0.01
        thr_full = 0.9 * bmax if bmax > 1.5 else 0.9
        r["brake_zero_%"] = float((b < thr_zero).mean() * 100)
        r["brake_full_%"] = float((b >= thr_full).mean() * 100)
        r["brake_mean"]   = float(b.mean())
        r["brake_max"]    = float(bmax)
    if tcol is not None:
        t = df[tcol].astype(float)
        r["throttle_mean"] = float(t.mean()); r["throttle_max"] = float(t.max())
    if bcol is not None and tcol is not None:
        b = df[bcol].astype(float); t = df[tcol].astype(float)
        bn = b / (b.max() if b.max() > 0 else 1); tn = t / (t.max() if t.max() > 0 else 1)
        r["coast_%"] = float(((bn < 0.05) & (tn < 0.05)).mean() * 100)
    if scol is not None:
        s = df[scol].astype(float)
        r["speed_mean"] = float(s.mean()); r["speed_std"] = float(s.std())
    return r, (bcol, tcol, scol)

if sac_monza and hum_monza:
    sdf = pd.read_parquet(sac_monza); hdf = pd.read_parquet(hum_monza)
    print(f"\nSAC kolon: {len(sdf.columns)} | Insan kolon: {len(hdf.columns)} | Ortak: {len(set(sdf.columns) & set(hdf.columns))}")
    sb, sc = behav(sdf); hb, hc = behav(hdf)
    print(f"Kullanilan kolonlar (brake,throttle,speed): SAC={sc} Insan={hc}")
    print(f"\n{'metrik':16s} {'SAC':>12s} {'INSAN':>12s}")
    for k in sorted(set(sb) | set(hb)):
        print(f"  {k:16s} {sb.get(k, float('nan')):>12.3f} {hb.get(k, float('nan')):>12.3f}")
    print("\n>> Degenerate adayi: SAC tarafinda ~0 / cok dusuk degerler (oran paydasi sifira gider)")
else:
    print("Monza ornegi eslesmedi - dosya adlarini kontrol et")

In [ ]:
# ============================================================
# SAC TESHIS 2 - 95-kolon semasi + populasyon saglik kontrolu
# ============================================================
import pandas as pd, numpy as np
from pathlib import Path

PROJECT = TRACK_ROOT
SAC_DIR = PROJECT / "data" / "reference_sac"
sac_files = sorted(SAC_DIR.rglob("*.parquet"))
monza = [f for f in sac_files if "monza" in f.name.lower()]
print(f"Monza SAC: {len(monza)} dosya")

# 1) 95 kolon semasi
s0 = pd.read_parquet(monza[0])
print(f"\nSAC kolon sayisi: {len(s0.columns)}")
print("Tum kolonlar:")
cols = list(s0.columns)
for i in range(0, len(cols), 5):
    print("  " + ", ".join(str(c) for c in cols[i:i+5]))

# 2) speed/brake/dist/id aday kolonlar
print("\n--- speed aday kolonlari (describe) ---")
spd_cols = [c for c in cols if any(k in str(c).lower() for k in ["speed","vel"])]
print("aday:", spd_cols)
for c in spd_cols:
    d = s0[c].astype(float)
    print(f"  {c:20s} min={d.min():.2f} mean={d.mean():.2f} max={d.max():.2f}")
dist_cols = [c for c in cols if any(k in str(c).lower() for k in ["dist"])]
print("dist aday:", dist_cols)
id_cols = [c for c in cols if any(k in str(c).lower() for k in ["driver","agent","id","name","sess","car","track"])]
print("id/meta aday:", id_cols)
for c in id_cols[:4]:
    try:
        print(f"  {c} -> {s0[c].iloc[0]}")
    except Exception:
        pass

# 3) populasyon saglik: per-dosya max speed dagilimi
SPD = "speed"  # 2. adimdan sonra gerekirse degistir
print(f"\n--- populasyon: per-dosya max '{SPD}' (ilk 80 Monza SAC) ---")
maxes, means = [], []
for f in monza[:80]:
    try:
        v = pd.read_parquet(f, columns=[SPD])[SPD].astype(float)
        maxes.append(float(v.max())); means.append(float(v.mean()))
    except Exception:
        pass
maxes = np.array(maxes); means = np.array(means)
if len(maxes):
    print(f"n={len(maxes)}")
    print(f"  per-dosya MAX speed : min={maxes.min():.2f} median={np.median(maxes):.2f} max={maxes.max():.2f}")
    print(f"  per-dosya MEAN speed: min={means.min():.2f} median={np.median(means):.2f} max={means.max():.2f}")
    thr = 30.0  # m/s varsayimi; gercek Monza lap max ~70 m/s
    print(f"  max<{thr} (degenerate/yavas aday): {int((maxes<thr).sum())}/{len(maxes)}")
    print(f"  max>={thr} (gercek lap aday)      : {int((maxes>=thr).sum())}/{len(maxes)}")

In [ ]:
# ============================================================
# SAC TESHIS 2b - array kolonlari atla + gecerlilik bayraklari + populasyon
# ============================================================
import pandas as pd, numpy as np
from pathlib import Path

PROJECT = TRACK_ROOT
SAC_DIR = PROJECT / "data" / "reference_sac"
monza = [f for f in sorted(SAC_DIR.rglob("*.parquet")) if "monza" in f.name.lower()]
print(f"Monza SAC: {len(monza)} dosya")
s0 = pd.read_parquet(monza[0])

def is_scalar_col(s):
    try:
        return np.ndim(s.iloc[0]) == 0
    except Exception:
        return False

array_cols = [c for c in s0.columns if not is_scalar_col(s0[c])]
print(f"\nArray-tipi kolonlar (atlanan): {array_cols}")

# 1) speed/velocity adaylari (sadece skaler)
print("\n--- speed/velocity adaylari (skaler) ---")
spd_cols = [c for c in s0.columns
            if any(k in str(c).lower() for k in ["speed", "vel"]) and is_scalar_col(s0[c])]
for c in spd_cols:
    d = pd.to_numeric(s0[c], errors="coerce")
    print(f"  {c:24s} min={d.min():.2f} mean={d.mean():.2f} max={d.max():.2f}")

# 2) gecerlilik / lap bayraklari (filtreleme icin)
print("\n--- gecerlilik bayraklari ---")
flag_cols = ["isInPit", "going_backwards", "LapInvalidated", "numberOfTyresOut",
             "out_of_track", "completedLaps", "LapCount", "LapDist", "NormalizedSplinePosition"]
for c in flag_cols:
    if c in s0.columns and is_scalar_col(s0[c]):
        d = pd.to_numeric(s0[c], errors="coerce")
        print(f"  {c:24s} min={d.min():.2f} mean={d.mean():.3f} max={d.max():.2f}  uniq={d.nunique()}")

# 3) populasyon saglik: per-dosya 'speed' (ilk 80 Monza)
print("\n--- populasyon: per-dosya 'speed' (ilk 80 Monza SAC) ---")
maxes, means = [], []
for f in monza[:80]:
    try:
        d = pd.to_numeric(pd.read_parquet(f, columns=["speed"])["speed"], errors="coerce")
        maxes.append(float(d.max())); means.append(float(d.mean()))
    except Exception:
        pass
maxes = np.array(maxes); means = np.array(means)
print(f"n={len(maxes)}")
print(f"  MAX speed : min={maxes.min():.2f} median={np.median(maxes):.2f} max={maxes.max():.2f}")
print(f"  MEAN speed: min={means.min():.2f} median={np.median(means):.2f} max={means.max():.2f}")
for thr in (30.0, 50.0):
    print(f"  per-dosya max<{thr:.0f}: {int((maxes < thr).sum())}/{len(maxes)}  (degenerate/yavas aday)")

In [ ]:
# ============================================================
# SAC TESHIS 3 - segment yapisi + LapDist kapsamasi + viraj basina ornek
# ============================================================
import pandas as pd, numpy as np
from pathlib import Path

PROJECT = TRACK_ROOT
SAC_DIR = PROJECT / "data" / "reference_sac"
monza = [f for f in sorted(SAC_DIR.rglob("*.parquet")) if "monza" in f.name.lower()]
print(f"Monza SAC toplam: {len(monza)}")

# tum listeye yayilmis ornek
N = 400
step = max(1, len(monza) // N)
sample = monza[::step]
print(f"Ornek: {len(sample)} dosya (stride={step})")

rows, spans, lo, hi, all_pos = [], [], [], [], []
for f in sample:
    try:
        d = pd.read_parquet(f, columns=["LapDist"])["LapDist"].astype(float)
        rows.append(len(d)); lo.append(d.min()); hi.append(d.max()); spans.append(d.max() - d.min())
        all_pos.append(d.values)
    except Exception:
        pass
rows = np.array(rows); spans = np.array(spans); lo = np.array(lo); hi = np.array(hi)
print(f"\nper-dosya satir : min={rows.min()} median={int(np.median(rows))} max={rows.max()}")
print(f"per-dosya span(m): min={spans.min():.1f} median={np.median(spans):.1f} max={spans.max():.1f}")
print(f"LapDist global  : min={lo.min():.0f} max={hi.max():.0f}  (Monza ~5793m)")

pos = np.concatenate(all_pos)
print(f"\nToplam SAC ornek satiri: {len(pos)}")
bins = np.arange(0, 6000, 200)
hist, _ = np.histogram(pos, bins=bins)
empty = [(int(bins[i]), int(bins[i+1])) for i in range(len(hist)) if hist[i] == 0]
print(f"Bos 200m bin (kapsama deligi): {len(empty)}/{len(hist)}")
if empty:
    print(f"  ornek bos araliklar: {empty[:8]}")

# corners_v3 viraj basina ornek
try:
    cv = pd.read_parquet(PROJECT / "data" / "features" / "monza_corners_v3.parquet")
    print(f"\ncorners_v3 kolonlari: {list(cv.columns)}")
    apex_col = next((c for c in cv.columns if "apex" in c.lower() and "dist" in c.lower()), None)
    if apex_col is None:
        apex_col = next((c for c in cv.columns if "dist" in c.lower()), cv.columns[0])
    print(f"apex kolonu: {apex_col} | viraj: {len(cv)}")
    for _, r in cv.iterrows():
        a = float(r[apex_col]); cnt = int(((pos >= a - 50) & (pos <= a + 50)).sum())
        print(f"  apex@{a:6.0f}m  +-50m ornek: {cnt}")
except Exception as e:
    print("corners_v3 okunamadi:", e)